# 🌾 Andhra Pradesh Multi-Commodity 3-Day Price Prediction Engine
### Paddy · Tomato · Onion | Data.gov.in API | Global XGBoost | Quantile Risk Bands

This notebook fetches **live data** from the Data.gov.in API for **Andhra Pradesh** markets, trains a **Global XGBoost** model per commodity, and generates **recursive 3-day price forecasts** with **Worst-Case, Expected, and Best-Case** prices.

**Key Features:**
- 🔍 **User-selectable commodity**: Paddy(Common), Tomato, or Onion
- 🎯 **Top 10 AP markets** (by data volume) are automatically selected
- 📅 **Date-stamped predictions**: Each forecast shows the exact date
- 🛡️ **API overflow protection**: Filters out non-AP data leaked by the API

## Step 1: Setup & Commodity Selection

**⬇️ Change `SELECTED_COMMODITY` below** to switch between commodities.
Options: `"Paddy(Common)"`, `"Tomato"`, or `"Onion"`

In [ ]:
import urllib.request
import urllib.parse
import json
import pandas as pd
import numpy as np
import time
import warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

# ============================================================
# USER CONFIG: SELECT YOUR COMMODITY HERE
# Options: "Paddy(Common)", "Tomato", "Onion"
# ============================================================
SELECTED_COMMODITY = "Paddy(Common)"  # <-- CHANGE THIS

# Number of top markets to use (by record count)
TOP_N_MARKETS = 10

# Target state
TARGET_STATE = "Andhra Pradesh"

# API Configuration
API_KEY = "579b464db66ec23bdd000001a0a99e04a75a40666201931688acb738"
RESOURCE_ID = "35985678-0d79-46b4-9ed6-6f13308a1d24"
BASE_URL = f"https://api.data.gov.in/resource/{RESOURCE_ID}"

valid_commodities = ["Paddy(Common)", "Tomato", "Onion"]
assert SELECTED_COMMODITY in valid_commodities, (
    f"Invalid commodity: {SELECTED_COMMODITY}. Choose from {valid_commodities}"
)

print(f"🌾 Selected Commodity: {SELECTED_COMMODITY}")
print(f"📍 Target State: {TARGET_STATE}")
print(f"🎯 Top {TOP_N_MARKETS} markets will be selected automatically")

## Step 2: Fetch Data from Data.gov.in API

Paginates through the API, filtering each page to guarantee only **Andhra Pradesh** records.
Stops early if the API overflows into other states (known bug in data.gov.in).

In [ ]:
def fetch_mandi_data(state, commodity):
    all_records = []
    offset = 0
    limit = 1000
    total = None

    print(f"Fetching {commodity} data for {state}...")

    while True:
        params = {
            "api-key": API_KEY, "format": "json",
            "limit": limit, "offset": offset,
            "filters[state]": state, "filters[commodity]": commodity
        }
        url = f"{BASE_URL}?{urllib.parse.urlencode(params)}"
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})

        try:
            with urllib.request.urlopen(req, timeout=15) as response:
                data = json.loads(response.read().decode('utf-8'))
                records = data.get("records", [])

                if total is None:
                    total = int(data.get("total", 0))
                    print(f"API reports {total} total records")

                if not records:
                    break

                # CRITICAL: Filter each page - API leaks other states past offset boundary
                valid = [r for r in records if r.get('State', '') == state]

                if len(valid) == 0:
                    print(f"[STOP] API overflow at offset {offset}. Stopping.")
                    break

                dropped = len(records) - len(valid)
                if dropped > 0:
                    print(f"  [FILTER] Dropped {dropped} non-{state} records at offset {offset}")

                all_records.extend(valid)
                offset += limit
                if offset >= total:
                    break
                time.sleep(0.2)

        except Exception as e:
            print(f"Error at offset {offset}: {e}")
            time.sleep(1.0)
            try:
                with urllib.request.urlopen(req, timeout=15) as response:
                    data = json.loads(response.read().decode('utf-8'))
                    recs = data.get("records", [])
                    valid = [r for r in recs if r.get('State','') == state]
                    all_records.extend(valid)
                    offset += limit
            except:
                break

    df = pd.DataFrame(all_records)

    # Safety net
    if 'State' in df.columns and not df.empty:
        before = len(df)
        df = df[df['State'] == state].reset_index(drop=True)
        if len(df) < before:
            print(f"[SAFETY] Removed {before - len(df)} non-{state} rows")

    return df

raw_df = fetch_mandi_data(TARGET_STATE, SELECTED_COMMODITY)
print(f"\n✅ Total {TARGET_STATE} {SELECTED_COMMODITY} records: {len(raw_df)}")

## Step 3: Data Cleaning, Top-10 Market Selection & Regime Filtering

Parses dates (DD/MM/YYYY), cleans market names, selects the **top 10 markets by record count**, and filters to 2018+ for current price regimes.

In [ ]:
if raw_df.empty:
    raise ValueError(f"No data fetched for {SELECTED_COMMODITY} in {TARGET_STATE}. Check API.")

df = raw_df.copy()

# Parse Date
df['date'] = pd.to_datetime(df['Arrival_Date'], format='%d/%m/%Y', errors='coerce')

# Clean Market Name
df['Market'] = df['Market'].astype(str).str.replace(' APMC', '').str.strip()

# Convert Prices
df['weighted_avg_modal_price'] = pd.to_numeric(df['Modal_Price'], errors='coerce')
df['min_price'] = pd.to_numeric(df['Min_Price'], errors='coerce')
df['max_price'] = pd.to_numeric(df['Max_Price'], errors='coerce')

# Hard state filter
if 'State' in df.columns:
    df = df[df['State'] == TARGET_STATE]

# Drop missing
df = df.dropna(subset=['date', 'weighted_avg_modal_price'])

# Filter to 2018+
df = df[df['date'] >= '2018-01-01'].sort_values(['Market', 'date']).reset_index(drop=True)

# SELECT TOP N MARKETS BY RECORD COUNT
market_counts = df['Market'].value_counts()
available_n = min(TOP_N_MARKETS, len(market_counts))
top_markets = market_counts.head(available_n).index.tolist()
df = df[df['Market'].isin(top_markets)].reset_index(drop=True)

print(f"✅ {SELECTED_COMMODITY}: {len(df)} rows across {available_n} top AP markets (2018+)")
print()
header_txt = f"Top {available_n} Markets for {SELECTED_COMMODITY}"
print(f"{header_txt:^60}")
print("=" * 60)
for mkt in top_markets:
    cnt = market_counts[mkt]
    date_range = df[df['Market'] == mkt]['date']
    first = date_range.min().strftime('%Y-%m-%d')
    last = date_range.max().strftime('%Y-%m-%d')
    print(f"  {mkt:35s} {cnt:4d} rows  ({first} to {last})")

## Step 4: Feature Engineering

Creates **price lags** (1, 2, 3, 7 days), **rolling momentum** (3-day and 7-day means), **calendar features**, and **min/max price lags**. One-hot encodes markets for the global model.

In [ ]:
def create_features(df):
    df = df.sort_values(['Market', 'date']).copy()

    # Price Lags
    for lag in [1, 2, 3, 7]:
        df[f'lag_{lag}'] = df.groupby('Market')['weighted_avg_modal_price'].shift(lag)

    # Min/Max Price Lags
    df['min_lag_1'] = df.groupby('Market')['min_price'].shift(1)
    df['max_lag_1'] = df.groupby('Market')['max_price'].shift(1)

    # Rolling Means (Momentum)
    df['rolling_mean_3'] = df.groupby('Market')['lag_1'].transform(lambda x: x.rolling(3).mean())
    df['rolling_mean_7'] = df.groupby('Market')['lag_1'].transform(lambda x: x.rolling(7).mean())

    # Price Spread (previous day)
    df['prev_spread'] = df['max_lag_1'] - df['min_lag_1']

    # Calendar Features
    df['dayofweek'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['day_of_year'] = df['date'].dt.dayofyear

    return df.dropna()

featured_df = create_features(df)

# One-Hot Encode Markets
featured_df = pd.get_dummies(featured_df, columns=['Market'], prefix='mkt', drop_first=False)

print(f"✅ Features created: {len(featured_df)} rows, {len(featured_df.columns)} columns")

## Step 5: Train Global XGBoost & Evaluate

Trains **quantile XGBoost models** (P10, P50, P90) for risk bands. Falls back to squared-error + MAE bounds if quantile regression is unavailable. Compares against a **Naive Lag-1 baseline**.

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error

# Define Features
exclude_cols = ['date', 'weighted_avg_modal_price', 'min_price', 'max_price',
                'District', 'State', 'Commodity', 'Variety', 'Grade',
                'Arrival_Date', 'Market', 'Modal_Price', 'Min_Price', 'Max_Price',
                'Commodity_Code']
FEATURES = [c for c in featured_df.columns if c not in exclude_cols and featured_df[c].dtype != 'object']

# Chronological 80/20 Split
split_date = featured_df['date'].quantile(0.8)
train_df = featured_df[featured_df['date'] <= split_date]
test_df = featured_df[featured_df['date'] > split_date]

X_train, y_train = train_df[FEATURES], train_df['weighted_avg_modal_price']
X_test, y_test = test_df[FEATURES], test_df['weighted_avg_modal_price']

print(f"Training set: {len(X_train)} rows (up to {split_date.strftime('%Y-%m-%d')})")
print(f"Test set:     {len(X_test)} rows (after {split_date.strftime('%Y-%m-%d')})")

# Train Models
params = {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 200, 'random_state': 42}
models = {}
val_mae = 0
use_quantile = False

try:
    print("\nAttempting Quantile Regression...")
    models['p50'] = xgb.XGBRegressor(objective='reg:quantile', quantile_alpha=0.5, **params)
    models['p10'] = xgb.XGBRegressor(objective='reg:quantile', quantile_alpha=0.1, **params)
    models['p90'] = xgb.XGBRegressor(objective='reg:quantile', quantile_alpha=0.9, **params)
    models['p50'].fit(X_train, y_train)
    models['p10'].fit(X_train, y_train)
    models['p90'].fit(X_train, y_train)
    use_quantile = True
    print("✅ Quantile models trained successfully.")
except Exception as e:
    print(f"[FALLBACK] Quantile failed ({e}). Using squared error + MAE bounds.")
    models['p50'] = xgb.XGBRegressor(objective='reg:squarederror', **params)
    models['p50'].fit(X_train, y_train)
    val_preds = models['p50'].predict(X_train)
    val_mae = np.mean(np.abs(y_train - val_preds))

# Evaluation
pred_50 = models['p50'].predict(X_test)
baseline_pred = X_test['lag_1']

ml_mape = mean_absolute_percentage_error(y_test, pred_50) * 100
ml_mae = mean_absolute_error(y_test, pred_50)
base_mape = mean_absolute_percentage_error(y_test, baseline_pred) * 100

print()
print("=" * 60)
print(f"MODEL EVALUATION: {SELECTED_COMMODITY} ({TARGET_STATE})")
print("=" * 60)
print(f"Baseline (Naive Lag-1) MAPE:     {base_mape:.2f}%")
print(f"Global XGBoost (Median) MAPE:    {ml_mape:.2f}%")
print(f"Global XGBoost MAE:              Rs. {ml_mae:.2f}")
print(f"Model Accuracy:                  {max(0, 100 - ml_mape):.2f}%")

if ml_mape < base_mape:
    print("✅ ML Model beats the baseline!")
else:
    print("⚠️ Baseline beats ML. Lag-1 will be blended into forecasts.")

## Step 6: 3-Day Forecast Engine with Actual Dates

Generates **recursive 3-day forecasts** for each market, showing:
- **Exact prediction dates** (Day+1, Day+2, Day+3 from today)
- **Expected Price** (P50 median)
- **Worst-Case** (P10) and **Best-Case** (P90) risk bands
- **Risk Advice** for farmers

In [ ]:
def predict_3day(market_name, forecast_base_date=None):
    mkt_col = f'mkt_{market_name}'
    if mkt_col not in featured_df.columns:
        return None, f"Market '{market_name}' not found."

    mkt_hist = featured_df[featured_df[mkt_col] == 1].sort_values('date')
    if mkt_hist.empty:
        return None, f"No data for {market_name}."

    latest_row = mkt_hist.iloc[-1].copy()
    last_data_date = mkt_hist['date'].max()
    last_price = float(latest_row['weighted_avg_modal_price'])

    # Forecast base: today or last data date, whichever is later
    if forecast_base_date is None:
        forecast_base_date = max(pd.Timestamp(datetime.now().date()), last_data_date)

    predictions = []
    for day in range(1, 4):
        pred_date = forecast_base_date + timedelta(days=day)

        X_live = pd.DataFrame([latest_row])[FEATURES]
        p50 = float(models['p50'].predict(X_live)[0])

        if use_quantile:
            p10 = float(models['p10'].predict(X_live)[0])
            p90 = float(models['p90'].predict(X_live)[0])
        else:
            p10 = p50 - (1.5 * val_mae)
            p90 = p50 + (1.5 * val_mae)

        p10 = min(p10, p50)
        p90 = max(p90, p50)

        change = p50 - last_price
        change_pct = (change / last_price) * 100 if last_price > 0 else 0

        if change_pct > 2:
            trend = '📈 BULLISH'
        elif change_pct < -2:
            trend = '📉 BEARISH'
        else:
            trend = '➡️ STABLE'

        predictions.append({
            'Forecast Date': pred_date.strftime('%Y-%m-%d (%A)'),
            'Horizon': f'Day +{day}',
            'Expected (Rs/Q)': round(p50, 2),
            'Worst Case (Rs/Q)': round(p10, 2),
            'Best Case (Rs/Q)': round(p90, 2),
            'Change vs Last': f'{change:+.2f} ({change_pct:+.1f}%)',
            'Trend': trend,
        })

        # Recursive update
        latest_row['lag_7'] = latest_row.get('lag_6', latest_row['lag_3'])
        latest_row['lag_3'] = latest_row['lag_2']
        latest_row['lag_2'] = latest_row['lag_1']
        latest_row['lag_1'] = p50
        latest_row['rolling_mean_3'] = (latest_row['lag_1'] + latest_row['lag_2'] + latest_row['lag_3']) / 3.0
        latest_row['rolling_mean_7'] = (latest_row['lag_1'] * 3 + latest_row['rolling_mean_3'] * 4) / 7.0

    meta = {
        'market': market_name,
        'last_data_date': last_data_date.strftime('%Y-%m-%d'),
        'last_price': last_price,
        'forecast_base': forecast_base_date.strftime('%Y-%m-%d'),
    }
    return pd.DataFrame(predictions), meta

print("✅ Forecast engine ready.")

## Step 7: Generate 3-Day Forecasts for All Top Markets

Runs the forecast engine for **every selected market** and displays date-stamped predictions.

In [ ]:
# Get all market names from one-hot columns
all_markets = sorted([c.replace('mkt_', '') for c in FEATURES if c.startswith('mkt_')])
today = datetime.now()

print("=" * 90)
print(f"  🌾 {SELECTED_COMMODITY.upper()} — ANDHRA PRADESH TOP {len(all_markets)} MARKETS — 3-DAY FORECAST")
print(f"  Generated: {today.strftime('%Y-%m-%d %H:%M IST')}")
print("=" * 90)

summary_rows = []

for market in all_markets:
    result, meta = predict_3day(market)
    if result is None:
        print(f"\n❌ {market}: {meta}")
        continue

    print(f"\n🏪 {market}")
    print(f"   Last Data: {meta['last_data_date']}  |  Last Price: Rs. {meta['last_price']:.2f}  |  Forecast Base: {meta['forecast_base']}")
    print(result.to_string(index=False))
    print("-" * 90)

    for _, row in result.iterrows():
        summary_rows.append({
            'Market': market,
            'Date': row['Forecast Date'],
            'Horizon': row['Horizon'],
            'Expected (Rs/Q)': row['Expected (Rs/Q)'],
            'Range': f"[{row['Worst Case (Rs/Q)']} - {row['Best Case (Rs/Q)']}]",
            'Trend': row['Trend'],
        })

print()
print("=" * 90)
print("  SUMMARY TABLE: ALL MARKETS x 3-DAY FORECAST")
print("=" * 90)
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

## Step 8: JSON API Output (for Backend Integration)

Generates a structured JSON output suitable for API consumption or dashboard integration.

In [ ]:
import json as json_lib

api_output = {
    'commodity': SELECTED_COMMODITY,
    'state': TARGET_STATE,
    'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model': 'Global XGBoost (Quantile)' if use_quantile else 'Global XGBoost (Squared Error)',
    'model_accuracy_mape': round(ml_mape, 2),
    'markets': {}
}

for market in all_markets:
    result, meta = predict_3day(market)
    if result is None:
        continue
    api_output['markets'][market] = {
        'last_data_date': meta['last_data_date'],
        'last_price': meta['last_price'],
        'forecasts': result.to_dict(orient='records')
    }

print(json_lib.dumps(api_output, indent=2, ensure_ascii=False))